# Reconciliação — Gold (Databricks) vs. KNIME

Compara os resultados da migração (Databricks Gold) com a saída do sistema legado
simulado (KNIME), por data, provando que a lógica de negócio foi preservada na
migração — não apenas o output de um sistema copiando o outro (ver ADR-01).

Critérios de comparação, por data de referência:
1. **Contagem de linhas**: nº de tickers processados em cada sistema.
2. **Valor do índice**: índice-proxy calculado pelo KNIME vs. pelo Databricks.

**Entrada:** CSVs do KNIME (`knime/indice_proxy_AAAA-MM-DD.csv`) via arquivos do
Workspace, e tabela `poc_b3_modernizacao.gold.indice_proxy`.
**Saída:** tabela `poc_b3_modernizacao.reconciliation.resultado` — status de cada
dia comparado (bate / diverge), com o detalhe da diferença quando houver.

In [0]:
%run ../setup/01_utilitarios_pipeline

In [0]:
# observabilidade - marca inicio da execucao
from datetime import datetime
inicio_execucao = datetime.now()

In [0]:
# widget - permite override manual da data (vazio = D-1 automatico)
dbutils.widgets.text("data_reconciliacao", "", "Data a reconciliar (AAAA-MM-DD, vazio = D-1 automatico)")

In [0]:
# imports
from pyspark.sql import functions as F

In [0]:
# descobre a data a reconciliar: override manual, ou D-1 automatico (ultima data disponivel na gold, anterior a hoje)
override = dbutils.widgets.get("data_reconciliacao").strip()

if override:
    DATA_RECONCILIACAO = override
    print(f"Data definida manualmente: {DATA_RECONCILIACAO}")
else:
    ultima_data_gold = (spark.table("poc_b3_modernizacao.gold.indice_proxy")
        .filter(F.col("data_referencia") < F.current_date())
        .agg(F.max("data_referencia"))
        .collect()[0][0]
    )
    if ultima_data_gold is None:
        raise ValueError("Nenhuma data anterior a hoje disponivel na gold.indice_proxy para reconciliar (D-1 automatico).")
    DATA_RECONCILIACAO = ultima_data_gold.strftime("%Y-%m-%d")
    print(f"Data D-1 detectada automaticamente: {DATA_RECONCILIACAO}")

In [0]:
# descobre o caminho do notebook atual, para localizar a pasta knime/ a partir dele
caminho_notebook = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
print(f"Caminho deste notebook: {caminho_notebook}")

In [0]:
# monta o caminho da pasta knime a partir do caminho do notebook
partes = caminho_notebook.split("/databricks/")
caminho_base_repo = partes[0]
caminho_knime = f"/Workspace{caminho_base_repo}/knime"

print(f"Caminho da pasta knime: {caminho_knime}")

# lista os arquivos disponiveis
arquivos_knime = dbutils.fs.ls(caminho_knime)
for a in arquivos_knime:
    print(f"  {a.name}")

In [0]:
# le os csvs do knime, converte indice para percentual e formata colunas para leitura
import re

def ler_csvs_knime(prefixo):
    arquivos = [a for a in arquivos_knime if a.name.startswith(prefixo)]
    dfs = []
    for arquivo in arquivos:
        data_match = re.search(r"(\d{4}-\d{2}-\d{2})", arquivo.name)
        data_referencia = data_match.group(1)

        if data_referencia != DATA_RECONCILIACAO:
            continue

        df_temp = (spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(arquivo.path)
            .withColumn("data_referencia", F.lit(data_referencia).cast("date"))
        )
        dfs.append(df_temp)

    if not dfs:
        return None

    df_final = dfs[0]
    for df in dfs[1:]:
        df_final = df_final.unionByName(df)
    return df_final

# indice-proxy: converte fracao para percentual (x100), 3 casas decimais
df_knime_indice = ler_csvs_knime("indice_proxy_")

if df_knime_indice is None:
    raise FileNotFoundError(f"CSV do KNIME nao encontrado para a data {DATA_RECONCILIACAO}. Verifique se o KNIME ja foi executado e commitado para esse dia.")

df_knime_indice = df_knime_indice.withColumn(
    "indice_proxy_pct_knime",
    F.round(F.col("Mean(retorno_diario)") * 100, 3)
).select("data_referencia", "indice_proxy_pct_knime")

# detalhado: retorno em percentual (3 casas) e data_hora mais enxuta (HH:mm)
df_knime_detalhado = ler_csvs_knime("cotacoes_detalhado_")
df_knime_detalhado = df_knime_detalhado.withColumn(
    "retorno_diario_pct_knime",
    F.round(F.col("retorno_diario") * 100, 3)
).withColumn(
    "hora_captura",
    F.date_format(F.col("data_hora"), "HH:mm")
).select("ticker", "data_referencia", "preco_atual", "fechamento_anterior", "retorno_diario_pct_knime", "hora_captura")

print("=== indice_proxy (KNIME) ===")
display(df_knime_indice)

print("=== cotacoes_detalhado (KNIME) ===")
display(df_knime_detalhado)

In [0]:
# le a gold do databricks, filtrando so a data a reconciliar
df_gold_indice = (spark.table("poc_b3_modernizacao.gold.indice_proxy")
    .filter(F.col("data_referencia") == DATA_RECONCILIACAO)
    .select("data_referencia", F.col("indice_proxy_pct").alias("indice_proxy_pct_databricks"))
)

df_gold_detalhado = (spark.table("poc_b3_modernizacao.gold.indicadores_diarios")
    .filter(F.col("data_referencia") == DATA_RECONCILIACAO)
    .select("ticker", "data_referencia", "preco_atual", "fechamento_anterior",
             F.col("retorno_diario_pct").alias("retorno_diario_pct_databricks"))
)

print("=== gold.indice_proxy (Databricks) ===")
display(df_gold_indice)

print("=== gold.indicadores_diarios (Databricks) ===")
display(df_gold_detalhado)

In [0]:
# compara indice-proxy: knime vs databricks
df_comparacao_indice = (df_knime_indice
    .join(df_gold_indice, on="data_referencia", how="outer")
    .withColumn(
        "diferenca_absoluta",
        F.round(F.abs(F.col("indice_proxy_pct_knime") - F.col("indice_proxy_pct_databricks")), 3)
    )
    .withColumn(
        "status",
        F.when(F.col("diferenca_absoluta") <= 0.01, F.lit("bate"))
         .otherwise(F.lit("diverge"))
    )
)

display(df_comparacao_indice)

In [0]:
# compara detalhado por ticker: knime vs databricks
df_comparacao_detalhado = (df_knime_detalhado
    .select("ticker", "data_referencia", "retorno_diario_pct_knime")
    .join(
        df_gold_detalhado.select("ticker", "data_referencia", "retorno_diario_pct_databricks"),
        on=["ticker", "data_referencia"],
        how="outer"
    )
    .withColumn(
        "diferenca_absoluta",
        F.round(F.abs(F.col("retorno_diario_pct_knime") - F.col("retorno_diario_pct_databricks")), 3)
    )
    .withColumn(
        "status",
        F.when(F.col("diferenca_absoluta") <= 0.01, F.lit("bate"))
         .otherwise(F.lit("diverge"))
    )
)

display(df_comparacao_detalhado.orderBy("ticker"))

In [0]:
# grava resultado da reconciliacao com causa raiz documentada
df_resultado_indice = df_comparacao_indice.withColumn(
    "causa_raiz",
    F.when(F.col("status") == "diverge",
           F.lit("Execucoes em horarios diferentes (KNIME ~17:22-17:24, Databricks 17:15 local) - mercado se moveu no intervalo. Nao indica falha de migracao de logica de negocio."))
     .otherwise(F.lit(None))
).withColumn("data_carga", F.current_timestamp())

df_resultado_detalhado = df_comparacao_detalhado.withColumn(
    "causa_raiz",
    F.when(F.col("status") == "diverge",
           F.lit("Execucoes em horarios diferentes - ver causa raiz agregada em reconciliation.resultado_indice."))
     .otherwise(F.lit(None))
).withColumn("data_carga", F.current_timestamp())

merge_ou_cria(df_resultado_indice, "poc_b3_modernizacao.reconciliation.resultado_indice", ["data_referencia"])
merge_ou_cria(df_resultado_detalhado, "poc_b3_modernizacao.reconciliation.resultado_detalhado", ["ticker", "data_referencia"])

In [0]:
# valida gravacao da reconciliacao
print("=== reconciliation.resultado_indice ===")
display(spark.table("poc_b3_modernizacao.reconciliation.resultado_indice"))

print("=== reconciliation.resultado_detalhado ===")
display(spark.table("poc_b3_modernizacao.reconciliation.resultado_detalhado").orderBy("ticker"))

In [0]:
# observabilidade - registra sucesso da execucao
data_referencia_reconciliacao = df_resultado_indice.agg(F.max("data_referencia")).collect()[0][0]

registrar_execucao(
    notebook="05_reconciliacao",
    data_referencia=data_referencia_reconciliacao,
    modo_execucao="reprocessamento_manual",
    status="sucesso",
    inicio=inicio_execucao,
    fim=datetime.now(),
)

In [0]:
display(spark.table("poc_b3_modernizacao.observability.pipeline_runs").orderBy("inicio"))

In [0]:
display(spark.table("poc_b3_modernizacao.reconciliation.resultado_indice").orderBy("data_referencia"))

In [0]:
print(f"DATA_RECONCILIACAO calculada nesta execucao: {DATA_RECONCILIACAO}")

In [0]:
# correcao - atualiza causa raiz das linhas afetadas pelo bug de cache do KNIME (27/08 a 01/09)
spark.sql("""
    UPDATE poc_b3_modernizacao.reconciliation.resultado_indice
    SET causa_raiz = 'CORRECAO (03/09): causa raiz original estava incorreta. O workflow KNIME estava com resultado em cache desde a primeira execucao real (27/08) ate 01/09 - o KNIME preserva estado de execucao junto com o workflow salvo, e "Execute all" nao forcava reexecucao dos nos ja marcados como prontos. O GET Request tambem estava afetado, nao so o calculo final - os precos capturados nesse periodo nao sao reais. Corrigido via Reset all + Execute all em 02/09. Esta linha NAO prova nem invalida a migracao: um dos dois lados da comparacao nunca teve dado real capturado.'
    WHERE to_date(data_referencia) BETWEEN '2026-08-27' AND '2026-09-01'
""")

spark.sql("""
    UPDATE poc_b3_modernizacao.reconciliation.resultado_detalhado
    SET causa_raiz = 'CORRECAO (03/09): ver causa raiz completa em reconciliation.resultado_indice - bug de cache do KNIME invalidou o dado deste dia.'
    WHERE to_date(data_referencia) BETWEEN '2026-08-27' AND '2026-09-01'
""")

print("Causas raiz corrigidas para o periodo afetado pelo bug de cache.")

In [0]:
display(spark.table("poc_b3_modernizacao.reconciliation.resultado_indice").orderBy("data_referencia"))